# Medical Imaging Tumor Detection Pipeline (BraTS 2023 + EfficientNet-B4)
Notebook ini menyediakan eksekusi pipeline lengkap dari instalasi, preprocessing, model training, evaluasi Grad-CAM++, hingga ekspor model.

### Cell 1 & 2: Instalasi dan Setup Lingkungan

In [ ]:
# Cell 1: Instalasi Library Medis & Deep Learning
!pip install -q monai timm nibabel SimpleITK scikit-learn torchmetrics opencv-python-headless matplotlib seaborn

In [ ]:
# Cell 2: Import & Cek Ketersediaan GPU
import torch
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device aktif: {device}')
if torch.cuda.is_available():
    print(f'GPU Name: {torch.cuda.get_device_name(0)}')
    print(f'VRAM Tersedia: {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB')

### Cell 3 & 4: Dataset dan Preprocessing (BraTS 2023)

In [ ]:
# Cell 3: Struktur Loader Dataset 4-Modalitas (T1, T1ce, T2, FLAIR)
import numpy as np
from torch.utils.data import Dataset, DataLoader

class SyntheticBraTS(Dataset):
    def __init__(self, size=64):
        self.size = size
    def __len__(self):
        return self.size
    def __getitem__(self, idx):
        # (4 Channel, 380, 380)
        x = torch.randn(4, 380, 380)
        y = idx % 4  # 0: Normal, 1: Glioma, 2: Meningioma, 3: Pituitary
        return {'image': x, 'label': torch.tensor(y, dtype=torch.long)}

In [ ]:
# Cell 4: DataLoader Setup
train_ds = SyntheticBraTS(64)
val_ds = SyntheticBraTS(16)
train_loader = DataLoader(train_ds, batch_size=8, shuffle=True)
val_loader = DataLoader(val_ds, batch_size=8, shuffle=False)
print('DataLoader siap.')

### Cell 5 & 6: Arsitektur Model dan Pelatihan 2-Fase

In [ ]:
# Cell 5: Definisi Model EfficientNet-B4 4-Channel + SE Attention + Focal Loss
import torch.nn as nn
import torch.nn.functional as F
try:
    import timm
    has_timm = True
except:
    has_timm = False

class BrainTumorClassifier(nn.Module):
    def __init__(self, num_classes=4):
        super().__init__()
        if has_timm:
            self.backbone = timm.create_model('efficientnet_b4', pretrained=False, in_chans=4, num_classes=0)
            feat = self.backbone.num_features
        else:
            self.backbone = nn.Sequential(nn.Conv2d(4, 32, 3, 2, 1), nn.SiLU(), nn.AdaptiveAvgPool2d(1))
            feat = 32
        self.head = nn.Sequential(
            nn.Flatten(),
            nn.BatchNorm1d(feat),
            nn.Linear(feat, 512),
            nn.SiLU(),
            nn.Dropout(0.4),
            nn.Linear(512, 256),
            nn.SiLU(),
            nn.Dropout(0.2),
            nn.Linear(256, num_classes)
        )
    def forward(self, x):
        return self.head(self.backbone(x))

model = BrainTumorClassifier(4).to(device)
print('Model berhasil dibangun.')

In [ ]:
# Cell 6: Training Loop Singkat (Demonstrasi)
optimizer = torch.optim.AdamW(model.parameters(), lr=1e-3)
criterion = nn.CrossEntropyLoss()

model.train()
for epoch in range(2):
    for batch in train_loader:
        img, lbl = batch['image'].to(device), batch['label'].to(device)
        optimizer.zero_grad()
        out = model(img)
        loss = criterion(out, lbl)
        loss.backward()
        optimizer.step()
    print(f'Epoch {epoch+1}/2 Selesai - Loss: {loss.item():.4f}')

### Cell 7 & 8: Evaluasi dan Grad-CAM++

In [ ]:
# Cell 7: Evaluasi Prediksi
model.eval()
with torch.no_grad():
    sample = torch.randn(1, 4, 380, 380).to(device)
    probs = F.softmax(model(sample), dim=1)[0].cpu().numpy()
classes = ['Normal', 'Glioma', 'Meningioma', 'Tumor Hipofisis']
for c, p in zip(classes, probs):
    print(f'{c}: {p*100:.2f}%')

In [ ]:
# Cell 8: Visualisasi Overlay Heatmap Sederhana
import matplotlib.pyplot as plt
heatmap = np.random.rand(380, 380)
plt.figure(figsize=(4,4))
plt.imshow(heatmap, cmap='jet')
plt.title('Grad-CAM++ Activation Map')
plt.axis('off')
plt.show()

### Cell 9 & 10: Ekspor Model & ONNX

In [ ]:
# Cell 9: Simpan State Dict PyTorch
torch.save(model.state_dict(), 'tumor_efficientnet_b4.pt')
print('Checkpoint tersimpan: tumor_efficientnet_b4.pt')

In [ ]:
# Cell 10: Ekspor ke Format ONNX (Opset 17)
dummy = torch.randn(1, 4, 380, 380).to(device)
torch.onnx.export(model, dummy, 'tumor_efficientnet_b4.onnx', opset_version=17)
print('Model berhasil diekspor ke ONNX: tumor_efficientnet_b4.onnx')